In [0]:
from pyspark.sql import functions as F

aug_df = spark.read.table("risknet_augmented_dataset")

print(f"Total Records : {aug_df.count():,}")
print(f"Total Columns : {len(aug_df.columns)}")

aug_df.printSchema()

Total Records : 1,000,000
Total Columns : 39
root
 |-- ApplicantID: string (nullable = true)
 |-- fraud_bool: integer (nullable = true)
 |-- income: double (nullable = true)
 |-- name_email_similarity: double (nullable = true)
 |-- prev_address_months_count: integer (nullable = true)
 |-- current_address_months_count: integer (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- days_since_request: double (nullable = true)
 |-- intended_balcon_amount: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- zip_count_4w: integer (nullable = true)
 |-- velocity_6h: double (nullable = true)
 |-- velocity_24h: double (nullable = true)
 |-- velocity_4w: double (nullable = true)
 |-- bank_branch_count_8w: integer (nullable = true)
 |-- date_of_birth_distinct_emails_4w: integer (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- credit_risk_score: integer (nullable = true)
 |-- email_is_free: integer (nullable = true)
 |-- housing_status: st

### Inference

The augmented dataset has been successfully loaded from the Delta table and is ready for graph node and relationship generation.

In [0]:
# Applicant Nodes

applicant_nodes_df = (
    aug_df.select(
        "ApplicantID",
        "credit_risk_score",
        "employment_status",
        "housing_status",
        "customer_age",
        "income"
    ).dropDuplicates(["ApplicantID"])
)

print(f"Applicant Nodes : {applicant_nodes_df.count():,}")

applicant_nodes_df.show(5, truncate=False)

Applicant Nodes : 1,000,000
+-----------+-----------------+-----------------+--------------+------------+------------------+
|ApplicantID|credit_risk_score|employment_status|housing_status|customer_age|income            |
+-----------+-----------------+-----------------+--------------+------------+------------------+
|APP00022913|174              |CB               |BC            |30          |0.6000000000000001|
|APP00237537|294              |CA               |BA            |40          |0.8               |
|APP00337556|73               |CE               |BE            |20          |0.7000000000000001|
|APP00349784|86               |CA               |BC            |50          |0.1               |
|APP00472072|187              |CA               |BB            |30          |0.1               |
+-----------+-----------------+-----------------+--------------+------------+------------------+
only showing top 5 rows


In [0]:
print("Total Applicant Nodes :", applicant_nodes_df.count())

print("Duplicate ApplicantIDs :",
      applicant_nodes_df.count() -
      applicant_nodes_df.select("ApplicantID").distinct().count())

print("Null ApplicantIDs :",
      applicant_nodes_df.filter(F.col("ApplicantID").isNull()).count())

Total Applicant Nodes : 1000000
Duplicate ApplicantIDs : 0
Null ApplicantIDs : 0


### Inference

Successfully created 1,000,000 unique Applicant nodes with demographic and financial attributes. Each ApplicantID is unique and will serve as the primary node identifier in the Neo4j graph.

In [0]:
# Create Alert Nodes
alert_nodes_df = (
    aug_df
    .filter(F.col("GenerateAlert") == True)
    .select(
        "AlertID",
        "RegistrationTimestamp",
        "is_false_positive_alert"
    )
    .dropDuplicates(["AlertID"])
)

In [0]:
# Validation
print("Total Alert Nodes :", alert_nodes_df.count())

print(
    "Duplicate AlertIDs :",
    alert_nodes_df.count() -
    alert_nodes_df.select("AlertID").distinct().count()
)

print(
    "Null AlertIDs :",
    alert_nodes_df.filter(F.col("AlertID").isNull()).count()
)

alert_nodes_df.show(5, truncate=False)

Total Alert Nodes : 109714
Duplicate AlertIDs : 0
Null AlertIDs : 0
+-----------+---------------------+-----------------------+
|AlertID    |RegistrationTimestamp|is_false_positive_alert|
+-----------+---------------------+-----------------------+
|ALT00074653|2025-04-25 11:08:40  |true                   |
|ALT00031906|2025-05-27 07:09:44  |true                   |
|ALT00066779|2025-02-10 09:58:51  |false                  |
|ALT00094282|2025-03-12 16:15:15  |true                   |
|ALT00000081|2025-04-18 14:32:50  |false                  |
+-----------+---------------------+-----------------------+
only showing top 5 rows


### Inference

Successfully created unique Alert nodes for all generated alerts. Each Alert node contains its identifier, registration timestamp, and false positive status, enabling temporal and investigative analysis within the Neo4j graph.

In [0]:
# Create Device Nodes
device_nodes_df = (
    aug_df
    .select("DeviceID")
    .dropDuplicates(["DeviceID"])
)

In [0]:
# Validation
print("Total Device Nodes :", device_nodes_df.count())

print(
    "Duplicate DeviceIDs :",
    device_nodes_df.count() -
    device_nodes_df.select("DeviceID").distinct().count()
)

print(
    "Null DeviceIDs :",
    device_nodes_df.filter(F.col("DeviceID").isNull()).count()
)

device_nodes_df.show(5, truncate=False)

Total Device Nodes : 938296
Duplicate DeviceIDs : 0
Null DeviceIDs : 0
+----------+
|DeviceID  |
+----------+
|LDEV298523|
|LDEV776724|
|LDEV320606|
|LDEV224996|
|LDEV473145|
+----------+
only showing top 5 rows


### Inference

Successfully created unique Device nodes from the augmented dataset. Each DeviceID represents a distinct device that can be linked to one or more applicants for graph-based fraud analysis.

In [0]:
# Create IP Nodes
ip_nodes_df = (
    aug_df
    .select("IPAddress")
    .dropDuplicates(["IPAddress"])
)

In [0]:
# Validation
print("Total IP Nodes :", ip_nodes_df.count())

print(
    "Duplicate IP Addresses :",
    ip_nodes_df.count() -
    ip_nodes_df.select("IPAddress").distinct().count()
)

print(
    "Null IP Addresses :",
    ip_nodes_df.filter(F.col("IPAddress").isNull()).count()
)

ip_nodes_df.show(5, truncate=False)

Total IP Nodes : 942832
Duplicate IP Addresses : 0
Null IP Addresses : 0
+--------------+
|IPAddress     |
+--------------+
|172.26.56.240 |
|172.28.95.85  |
|172.16.91.74  |
|172.17.211.119|
|172.24.70.149 |
+--------------+
only showing top 5 rows


### Inference

Successfully created unique IP nodes from the augmented dataset. Each IPAddress represents a distinct network identity that can be connected to multiple applicants for graph-based fraud detection.

In [0]:
# Create Applicant → Device Relationships
applicant_device_rel_df = (
    aug_df
    .select("ApplicantID", "DeviceID")
    .dropDuplicates()
)

In [0]:
# Validation
print("Total Applicant-Device Relationships :", applicant_device_rel_df.count())

print(
    "Duplicate Relationships :",
    applicant_device_rel_df.count() -
    applicant_device_rel_df.distinct().count()
)

print(
    "Null ApplicantIDs :",
    applicant_device_rel_df.filter(F.col("ApplicantID").isNull()).count()
)

print(
    "Null DeviceIDs :",
    applicant_device_rel_df.filter(F.col("DeviceID").isNull()).count()
)

applicant_device_rel_df.show(5, truncate=False)

Total Applicant-Device Relationships : 1000000
Duplicate Relationships : 0
Null ApplicantIDs : 0
Null DeviceIDs : 0
+-----------+----------+
|ApplicantID|DeviceID  |
+-----------+----------+
|APP00017719|LDEV215521|
|APP00407226|LDEV290007|
|APP00557250|LDEV771610|
|APP00649614|LDEV781250|
|APP00826905|LDEV545006|
+-----------+----------+
only showing top 5 rows


### Inference

Successfully created USES_DEVICE relationships linking applicants to their registered devices. These relationships enable identification of multiple applicants sharing the same device, supporting graph-based fraud detection.

In [0]:
# Create Applicant → IP Relationships
applicant_ip_rel_df = (
    aug_df
    .select("ApplicantID", "IPAddress")
    .dropDuplicates()
)

In [0]:
# Validation
print("Total Applicant-IP Relationships :", applicant_ip_rel_df.count())

print(
    "Duplicate Relationships :",
    applicant_ip_rel_df.count() -
    applicant_ip_rel_df.distinct().count()
)

print(
    "Null ApplicantIDs :",
    applicant_ip_rel_df.filter(F.col("ApplicantID").isNull()).count()
)

print(
    "Null IP Addresses :",
    applicant_ip_rel_df.filter(F.col("IPAddress").isNull()).count()
)

applicant_ip_rel_df.show(5, truncate=False)

Total Applicant-IP Relationships : 1000000
Duplicate Relationships : 0
Null ApplicantIDs : 0
Null IP Addresses : 0
+-----------+--------------+
|ApplicantID|IPAddress     |
+-----------+--------------+
|APP00089400|172.16.153.197|
|APP00135073|172.17.75.98  |
|APP00260277|172.19.50.172 |
|APP00310238|172.19.245.158|
|APP00499872|172.22.215.129|
+-----------+--------------+
only showing top 5 rows


Inference

Successfully created USES_IP relationships linking applicants to their registration IP addresses. These relationships enable identification of multiple applicants sharing the same IP address, supporting graph-based fraud detection.

In [0]:
# Create Alert → Applicant Relationships
alert_applicant_rel_df = (
    aug_df
    .filter(F.col("GenerateAlert") == True)
    .select("AlertID", "ApplicantID")
    .dropDuplicates()
)

In [0]:
# Validation
print("Total Alert-Applicant Relationships :", alert_applicant_rel_df.count())

print(
    "Duplicate Relationships :",
    alert_applicant_rel_df.count() -
    alert_applicant_rel_df.distinct().count()
)

print(
    "Null AlertIDs :",
    alert_applicant_rel_df.filter(F.col("AlertID").isNull()).count()
)

print(
    "Null ApplicantIDs :",
    alert_applicant_rel_df.filter(F.col("ApplicantID").isNull()).count()
)

alert_applicant_rel_df.show(5, truncate=False)

Total Alert-Applicant Relationships : 109714
Duplicate Relationships : 0
Null AlertIDs : 0
Null ApplicantIDs : 0
+-----------+-----------+
|AlertID    |ApplicantID|
+-----------+-----------+
|ALT00038735|APP00352307|
|ALT00104984|APP00958218|
|ALT00043087|APP00391646|
|ALT00064898|APP00592236|
|ALT00090517|APP00825457|
+-----------+-----------+
only showing top 5 rows


### Inference

Successfully created GENERATED_FOR relationships linking each generated alert to its corresponding applicant. These relationships enable efficient investigation of flagged applicants and support graph-based fraud analysis.

In [0]:
# Validation
print("========== NODE COUNTS ==========")
print("Applicant Nodes :", applicant_nodes_df.count())
print("Device Nodes    :", device_nodes_df.count())
print("IP Nodes        :", ip_nodes_df.count())
print("Alert Nodes     :", alert_nodes_df.count())

print("\n========== RELATIONSHIP COUNTS ==========")
print("Applicant → Device :", applicant_device_rel_df.count())
print("Applicant → IP     :", applicant_ip_rel_df.count())
print("Alert → Applicant  :", alert_applicant_rel_df.count())

========== NODE COUNTS ==========
Applicant Nodes : 1000000
Device Nodes    : 938296
IP Nodes        : 942832
Alert Nodes     : 109714

========== RELATIONSHIP COUNTS ==========
Applicant → Device : 1000000
Applicant → IP     : 1000000
Alert → Applicant  : 109714


### Inference

Successfully validated all node and relationship DataFrames. The generated graph data is complete, consistent, and ready for export as Neo4j-compatible CSV files.

In [0]:
# Export Neo4j CSV Files
EXPORT_PATH = "/Volumes/risknet_catalog/risknet/risknet_volume/graph_export"

# Applicant Nodes
(applicant_nodes_df
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{EXPORT_PATH}/applicant_nodes"))

# Device Nodes
(device_nodes_df
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{EXPORT_PATH}/device_nodes"))

# IP Nodes
(ip_nodes_df
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{EXPORT_PATH}/ip_nodes"))

# Alert Nodes
(alert_nodes_df
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{EXPORT_PATH}/alert_nodes"))

# Applicant → Device Relationships
(applicant_device_rel_df
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{EXPORT_PATH}/applicant_device_relationships"))

# Applicant → IP Relationships
(applicant_ip_rel_df
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{EXPORT_PATH}/applicant_ip_relationships"))

# Alert → Applicant Relationships
(alert_applicant_rel_df
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(f"{EXPORT_PATH}/alert_applicant_relationships"))


In [0]:
display(dbutils.fs.ls(EXPORT_PATH))

path,name,size,modificationTime
dbfs:/Volumes/risknet_catalog/risknet/risknet_volume/graph_export/alert_applicant_relationships/,alert_applicant_relationships/,0,1785484139536
dbfs:/Volumes/risknet_catalog/risknet/risknet_volume/graph_export/alert_nodes/,alert_nodes/,0,1785484139536
dbfs:/Volumes/risknet_catalog/risknet/risknet_volume/graph_export/applicant_device_relationships/,applicant_device_relationships/,0,1785484139536
dbfs:/Volumes/risknet_catalog/risknet/risknet_volume/graph_export/applicant_ip_relationships/,applicant_ip_relationships/,0,1785484139536
dbfs:/Volumes/risknet_catalog/risknet/risknet_volume/graph_export/applicant_nodes/,applicant_nodes/,0,1785484139536
dbfs:/Volumes/risknet_catalog/risknet/risknet_volume/graph_export/device_nodes/,device_nodes/,0,1785484139536
dbfs:/Volumes/risknet_catalog/risknet/risknet_volume/graph_export/ip_nodes/,ip_nodes/,0,1785484139537
